In [ ]:
import numpy as np

class GridWorld:
    def __init__(self):
        self.rows, self.cols = 3, 4
        self.wall = (1, 1)
        self.terminals = {(0, 3): 1, (1, 3): -1}
        self.actions = ["U", "D", "L", "R"]
        self.gamma = 0.9
        self.step = -0.01

    def is_terminal(self, s):
        return s in self.terminals

    def move(self, s, a):
        if self.is_terminal(s): return s
        r, c = s
        if a == "U": r -= 1
        if a == "D": r += 1
        if a == "L": c -= 1
        if a == "R": c += 1
        ns = (r, c)
        if r < 0 or r >= self.rows or c < 0 or c >= self.cols or ns == self.wall:
            return s
        return ns


def value_iteration(env, theta=1e-5):
    V = np.zeros((env.rows, env.cols))

    while True:
        delta = 0
        V_old = V.copy()

        for r in range(env.rows):
            for c in range(env.cols):
                s = (r, c)

                if s == env.wall: 
                    continue
                if env.is_terminal(s):
                    V[r, c] = env.terminals[s]
                    continue

                values = []
                for a in env.actions:
                    ns = env.move(s, a)
                    if env.is_terminal(ns):
                        v = env.step + env.gamma * env.terminals[ns]
                    else:
                        nr, nc = ns
                        v = env.step + env.gamma * V_old[nr, nc]
                    values.append(v)

                new_v = max(values)
                delta = max(delta, abs(new_v - V[r, c]))
                V[r, c] = new_v

        if delta < theta:
            break

    return V


def extract_policy(env, V):
    P = np.full((env.rows, env.cols), "", dtype=object)

    for r in range(env.rows):
        for c in range(env.cols):
            s = (r, c)

            if s == env.wall: P[r, c] = "#"; continue
            if env.is_terminal(s): P[r, c] = "T"; continue

            best_a, best_v = None, -1e9
            for a in env.actions:
                ns = env.move(s, a)
                if env.is_terminal(ns):
                    v = env.terminals[ns]
                else:
                    nr, nc = ns
                    v = env.step + env.gamma * V[nr, nc]
                if v > best_v:
                    best_v, best_a = v, a

            P[r, c] = best_a

    return P


env = GridWorld()
V = value_iteration(env)
policy = extract_policy(env, V)

print("Value Function:\n", V)
print("\nPolicy:\n", policy)

Value Function:
[[ 0.7019    0.791     0.89      1.      ]
 [ 0.62171   0.        0.791    -1.      ]
 [ 0.549539  0.62171   0.7019    0.62171 ]]

Optimal Policy:
[['R' 'R' 'R' 'T']
 ['U' '#' 'U' 'T']
 ['U' 'R' 'U' 'L']]
